In [196]:
import IPython
import numpy as np
import pyopenjtalk
import pyworld as pw
import tensorflow as tf
import ttslearn
from nnmnkwii.io import hts
from nnmnkwii.frontend import merlin

In [197]:
duration_model = tf.keras.models.load_model("duration_model.h5")

In [198]:
acoustic_model = tf.keras.models.load_model("model1.h5")

In [231]:
text = "吾輩は猫である。名前はまだ無い。"

contexts = pyopenjtalk.extract_fullcontext(text)
labels = hts.HTSLabelFile.create_from_contexts(contexts)

In [232]:
binary_dict, numeric_dict = hts.load_question_set(ttslearn.util.example_qst_file())
lng = merlin.linguistic_features(
    labels,
    binary_dict,
    numeric_dict,
)

In [233]:
mean, std = np.load("generated/linguistic/mean.npy"), np.load("generated/linguistic/std.npy")

In [234]:
lng = np.divide(
            lng - np.expand_dims(mean, axis=0),
            np.expand_dims(std, axis=0),
            out=np.zeros_like(lng),
            where=(std != 0),
        )

In [235]:
lng

array([[ 0.        ,  0.        ,  0.        , ..., -1.56541074,
         0.40133428,  0.31531413],
       [ 0.        ,  0.        ,  0.        , ..., -0.52895883,
         0.86331912,  0.08964204],
       [ 0.        ,  0.        ,  0.        , ..., -0.52895883,
         0.86331912,  0.08964204],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -1.07445983,
        -0.98462024, -0.92588237],
       [ 0.        ,  0.        ,  0.        , ..., -1.07445983,
        -0.98462024, -0.92588237],
       [ 0.        ,  0.        ,  0.        , ..., -1.56541074,
        -0.98462024, -0.92588237]])

In [236]:
durations_normalized = duration_model.predict(lng)

2/2 [==============================] - 0s 1ms/step


In [237]:
dur_mean, dur_std = np.load("generated/duration/mean.npy"), np.load("generated/duration/std.npy")
durations = dur_mean + dur_std * durations_normalized
durations

array([[59.95352  ],
       [10.748922 ],
       [ 9.069849 ],
       [13.027321 ],
       [ 8.914478 ],
       [20.37425  ],
       [12.688684 ],
       [ 8.196058 ],
       [19.727833 ],
       [ 7.747322 ],
       [12.150566 ],
       [13.344731 ],
       [15.246687 ],
       [11.338182 ],
       [ 9.190653 ],
       [ 4.519552 ],
       [21.512623 ],
       [11.099047 ],
       [18.748888 ],
       [42.969574 ],
       [12.939634 ],
       [ 8.332512 ],
       [16.818453 ],
       [18.22062  ],
       [ 6.5429506],
       [22.19858  ],
       [ 8.260279 ],
       [17.830917 ],
       [10.911297 ],
       [11.287702 ],
       [10.420998 ],
       [14.9394455],
       [21.051655 ],
       [21.994812 ],
       [54.63307  ]], dtype=float32)

In [238]:
labels.set_durations(durations)

In [239]:
lng_frame = merlin.linguistic_features(
    labels,
    binary_dict,
    numeric_dict,
    add_frame_features=True, subphone_features="coarse_coding",
)

In [240]:
lng_frame.shape

(586, 329)

In [241]:
lng_f_mean, lng_f_std = np.load("generated/linguistic_frame/mean.npy"), np.load("generated/linguistic_frame/std.npy")

lng_frame_normalized = np.divide(
            lng_frame - np.expand_dims(lng_f_mean, axis=0),
            np.expand_dims(lng_f_std, axis=0),
            out=np.zeros_like(lng_frame),
            where=(lng_f_std != 0),
        )

In [242]:
aco_normalized = acoustic_model.predict(lng_frame_normalized)

19/19 [==============================] - 0s 2ms/step


In [243]:
aco_normalized = np.concatenate(aco_normalized, axis=-1)
aco_normalized.shape

(586, 1027)

In [244]:
aco_mean, aco_std = np.load("generated/acoustic/mean.npy"), np.load("generated/acoustic/std.npy")
aco = aco_mean + aco_std * aco_normalized

In [245]:
aco

array([[ 1.1110687e+00,  1.1215091e-02, -6.8642274e-03, ...,
         1.0000210e+00,  1.0000271e+00,  1.0000000e+00],
       [ 2.2912140e+00,  1.2470260e-02, -5.0635673e-03, ...,
         1.0000153e+00,  1.0000237e+00,  1.0000000e+00],
       [ 1.5189209e+00,  1.2848254e-02, -4.5212954e-03, ...,
         1.0000570e+00,  1.0000445e+00,  1.0000000e+00],
       ...,
       [ 2.6656799e+00,  1.0957081e-02, -7.2343647e-03, ...,
         9.9996305e-01,  9.9999440e-01,  1.0000000e+00],
       [ 2.3250122e+00,  1.0986045e-02, -7.1928203e-03, ...,
         9.9997467e-01,  1.0000004e+00,  1.0000000e+00],
       [ 4.8436966e+00,  1.1236180e-02, -6.8339705e-03, ...,
         9.9997842e-01,  1.0000024e+00,  1.0000000e+00]], dtype=float32)

In [246]:
audio = pw.synthesize(aco[:, 0].astype(np.float64), aco[:, 1:514].astype(np.float64), aco[:, 514:].astype(np.float64), 22050)
IPython.display.Audio(audio, rate=22050)